## 1. Install dependencies


In [ ]:
!pip install -q langchain-groq pydantic

In [ ]:
"""
═══════════════════════════════════════════════════════════════════════
 TUTORIAL DEMO — LLM AS A JUDGE (Evaluation Design Patterns)
 Companion code for the conference tutorial:
   "Choosing LLM as a Judge: Evaluation Design Patterns for Production GenAI"
═══════════════════════════════════════════════════════════════════════

 USE CASE
   Customer-support RAG bot for a SaaS product.
   We have a small "golden set" of (question, ground_truth, context) and
   two candidate response generators (v1 = baseline, v2 = improved).
   Goal: decide which to ship + catch regressions.

 MAPPING TO TUTORIAL SECTIONS
   Section 1  Evaluation Challenges        →  intro_why_eval_is_hard()
   Section 2  What is LLM-as-a-Judge?      →  intro_what_is_a_judge()
   Section 3  Core Components              →  PROMPT TEMPLATES + _parse_json
   Section 4  Design Patterns              →  Patterns A–E below
   Section 5  Scoring & Calibration        →  calibration_report() + stability_check()
   Section 6  Use Cases                    →  run_full_eval() + regression_report()
   Section 7  Failure Modes & Trade-offs   →  failure_mode_demo()
   Section 8  Wrap-up                      →  ship_decision() + when_NOT_to_use()

 PATTERNS DEMONSTRATED
   A. Pass/Fail judge          — binary "is this answer acceptable to ship?"
   B. Rubric scoring (1-5)     — multi-criterion structured judgment + CoT
   C. Pairwise ranking         — "which of A/B is better?" with position-swap
   D. Reference-based judge    — answer vs ground truth   (correctness)
   E. Reference-FREE judge     — answer vs retrieved ctx  (faithfulness, no GT)

 Run:
   set GROQ_API_KEY=gsk_...   (or use Colab Secrets / .env)
   python demo_llm_as_judge.py
═══════════════════════════════════════════════════════════════════════
"""

## 2. Set your Groq API key
Get a free key at https://console.groq.com/keys

In [ ]:
import os, json, time, statistics, random
from dataclasses import dataclass, asdict
from typing import Literal

# --- .env support (no-op if python-dotenv not installed) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# --- GROQ_API_KEY: Colab Secrets → fallback to manual prompt ---
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    import getpass
    if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
        print("✅ GROQ_API_KEY set via prompt")

from langchain_groq import ChatGroq

# Two judge models (using the same backend with different temperatures
# to keep the demo single-provider; in production you'd use a stronger
# model — e.g. GPT-4-class — as the judge).
judge_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)
gen_llm   = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
print("✅ Loaded judge + generator models\n")


✅ GROQ_API_KEY loaded from Colab Secrets
✅ Loaded judge + generator models



## 3. Imports + load models
Judge is **stronger** than the generator (avoids self-confirmation bias).

In [ ]:
import json, time, statistics, random, re
from dataclasses import dataclass
from langchain_groq import ChatGroq

random.seed(0)

# QUICK = True trims the golden set for fast on-stage runs. Set False for full 6 items.
QUICK = True


judge_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0, timeout=30)
gen_llm   = ChatGroq(model="llama-3.1-8b-instant",    temperature=0.7, timeout=30)
print(f"Loaded judge=llama-3.3-70b (T=0)  generator=llama-3.1-8b (T=0.7)  QUICK={QUICK}")

Loaded judge=llama-3.3-70b (T=0)  generator=llama-3.1-8b (T=0.7)  QUICK=True


## 4. Golden set (hand-curated, with human labels)
6 customer-support Q/A items. **q6 is a trap** - context doesn't answer the question; a good judge must fail any fabricated answer.


In [ ]:
GOLDEN = [
    {"id": "q1",
     "question": "How do I reset my password?",
     "context": "To reset your password, go to Settings -> Security -> Reset password. A verification email will be sent.",
     "ground_truth": "Open Settings -> Security -> Reset password; you will receive a verification email.",
     "human_label_passfail": "pass"},
    {"id": "q2",
     "question": "What's the price of the Enterprise plan?",
     "context": "The Enterprise plan starts at $499/month with custom pricing for >100 seats. Contact sales for a quote.",
     "ground_truth": "The Enterprise plan starts at $499/month; contact sales for >100-seat custom pricing.",
     "human_label_passfail": "pass"},
    {"id": "q3",
     "question": "Can I export my data to CSV?",
     "context": "Data export is supported in JSON and CSV formats from Settings -> Export. Free tier is limited to JSON.",
     "ground_truth": "Yes - CSV export is supported from Settings -> Export (JSON only on the free tier).",
     "human_label_passfail": "pass"},
    {"id": "q4",
     "question": "Do you offer SOC 2 compliance?",
     "context": "We are SOC 2 Type II certified. Reports are available on request from compliance@example.com.",
     "ground_truth": "Yes - we are SOC 2 Type II certified; request the report from compliance@example.com.",
     "human_label_passfail": "pass"},
    {"id": "q5",
     "question": "How do I cancel my subscription?",
     "context": "Cancel anytime from Settings -> Billing -> Cancel. Access continues until the end of the billing period.",
     "ground_truth": "Settings -> Billing -> Cancel. You keep access until the end of the current billing period.",
     "human_label_passfail": "pass"},
    # Negative example - context does NOT answer the question.
    {"id": "q6",
     "question": "What is your refund policy?",
     "context": "Our Pro plan is billed monthly at $29/month and includes unlimited projects and email support.",
     "ground_truth": "The provided context does not describe a refund policy; please contact support.",
     "human_label_passfail": "fail"},
]
print(f"Loaded {len(GOLDEN)} golden items")

Loaded 6 golden items


## 5. Helpers — safe LLM call + JSON parse
Production judges fail to return valid JSON ~5-15% of the time. Never let that crash the pipeline.

In [ ]:
def _parse_json(raw, default):
    """Robust JSON extractor: handles markdown ```json fences, prose wrappers,
    and trailing commas. Returns a copy of `default` on any failure."""
    if not isinstance(raw, str) or not raw.strip():
        print("   [warn] parse fallback: empty LLM output")
        return dict(default)

    text = raw.strip()

    # 1. Strip ```json ... ``` or ``` ... ``` fences if present.
    fence = re.search(r"```(?:json)?\s*(.+?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        text = fence.group(1).strip()

    # 2. Try direct parse first.
    try:
        return json.loads(text)
    except Exception:
        pass

    # 3. Find first balanced {...} block by brace-counting (handles nested objects).
    start = text.find("{")
    if start < 0:
        print("   [warn] parse fallback: no JSON braces")
        return dict(default)
    depth, end = 0, -1
    for i in range(start, len(text)):
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    if end < 0:
        print("   [warn] parse fallback: unbalanced braces")
        return dict(default)

    candidate = text[start:end]
    # 4. Strip trailing commas before } or ] (common LLM mistake).
    candidate = re.sub(r",(\s*[}\]])", r"\1", candidate)

    try:
        return json.loads(candidate)
    except Exception as ex:
        print(f"   [warn] parse fallback: {type(ex).__name__}: {str(ex)[:60]}")
        return dict(default)


def _safe_invoke(llm, prompt, retries=1, timeout_s=30.0):
    last_err = None
    for attempt in range(retries + 1):
        try:
            t0 = time.time()
            out = llm.invoke(prompt).content
            if time.time() - t0 > timeout_s:
                print(f"   [warn] slow response ({time.time()-t0:.1f}s)")
            return out or ""
        except Exception as ex:
            last_err = ex
            print(f"   [warn] LLM error (attempt {attempt+1}): {type(ex).__name__}")
            time.sleep(0.5)
    print(f"   [ERROR] LLM failed after {retries+1} attempts: {last_err}")
    return ""


def banner(title):
    print("\n" + "="*70 + f"\n  {title}\n" + "="*70)

## 6. Two response generators — v1 baseline vs v2 improved
v1 is deliberately weak (truncated, no grounding guardrail) so the patterns visibly react.

In [ ]:
def gen_v1(q, ctx):
    """Baseline - terse, no grounding guardrail."""
    p = f"Answer the question in one short sentence.\n\nContext: {ctx}\n\nQuestion: {q}\nAnswer:"
    out = (_safe_invoke(gen_llm, p) or "(generation failed)").strip()
    return out[:80]

def gen_v2(q, ctx):
    """Improved - grounded, complete."""
    p = (f"You are a helpful support assistant. Using ONLY the context, "
         f"give a complete, accurate answer in 1-3 sentences. Do not invent.\n\n"
         f"Context: {ctx}\n\nQuestion: {q}\nAnswer:")
    return (_safe_invoke(gen_llm, p) or "(generation failed)").strip()

## 7. Pattern A — Pass/Fail judge
Binary ship/no-ship verdict. Used for CI gates and canaries.

In [ ]:
PASSFAIL_PROMPT = """You are an evaluator for a customer-support bot.
Decide whether the ANSWER is acceptable to ship to a real user.

Reject (FAIL) if any of:
- Factually wrong vs the CONTEXT
- Invents information not in the CONTEXT
- Misses a critical part of the question
- Tone is rude / unprofessional

Reply with ONLY a JSON object. The "verdict" field must be the string "pass" or "fail".
Example of the exact shape (do not copy the values):
{{"verdict": "pass", "reason": "answer is grounded and complete"}}

Now evaluate:
QUESTION: {q}
CONTEXT:  {ctx}
ANSWER:   {a}
"""

def judge_passfail(q, ctx, a):
    raw = _safe_invoke(judge_llm, PASSFAIL_PROMPT.format(q=q, ctx=ctx, a=a))
    return _parse_json(raw, default={"verdict": "fail", "reason": "parse error"})

## 8. Pattern B — Rubric scoring (1–5 on 4 dimensions)
Tells you *which* axis regressed: faithfulness, completeness, relevance, clarity.


In [ ]:
RUBRIC_PROMPT = """You are a strict evaluator. Score the ANSWER on 4 dimensions
using this rubric. For EACH score also write a 1-line justification.

Rubric (1=very poor, 3=acceptable, 5=excellent):
  - faithfulness  : answer is supported by the CONTEXT, no fabrication
  - completeness  : answer covers all parts of the QUESTION
  - relevance     : answer directly addresses the QUESTION
  - clarity       : concise, well-written, professional tone

Reply with ONLY a JSON object. All four scores must be INTEGERS between 1 and 5.
Example of the exact shape (do not copy the values):
{{"reasoning": "the answer is grounded but a bit terse", "scores": {{"faithfulness": 5, "completeness": 3, "relevance": 5, "clarity": 4}}, "justifications": {{"faithfulness": "all facts present in context", "completeness": "missed one sub-question", "relevance": "directly on-topic", "clarity": "clear and professional"}}}}

Now evaluate:
QUESTION: {q}
CONTEXT:  {ctx}
ANSWER:   {a}
"""

def judge_rubric(q, ctx, a):
    raw = _safe_invoke(judge_llm, RUBRIC_PROMPT.format(q=q, ctx=ctx, a=a))
    return _parse_json(raw, default={
        "scores": {"faithfulness": 0, "completeness": 0, "relevance": 0, "clarity": 0},
        "reasoning": "parse error", "justifications": {},
    })

## 9. Pattern C — Pairwise ranking with position-swap
LLMs prefer "Answer A" simply because it's first. Swap positions and require agreement.

In [ ]:
PAIRWISE_PROMPT = """You are comparing two candidate ANSWERS to the same QUESTION.
Pick the better one. If they are essentially equal, say "tie".

Criteria: faithful to CONTEXT > complete > clear.

Reply with ONLY a JSON object. "winner" must be exactly one of: "A", "B", or "tie".
Example of the exact shape (do not copy the values):
{{"winner": "A", "reason": "answer A is more faithful to the context"}}

Now compare:
QUESTION: {q}
CONTEXT:  {ctx}

ANSWER A: {a}
ANSWER B: {b}
"""

def judge_pairwise(q, ctx, a, b):
    """Run twice with positions swapped to neutralize position bias."""
    r1 = _parse_json(_safe_invoke(judge_llm, PAIRWISE_PROMPT.format(q=q, ctx=ctx, a=a, b=b)),
                     default={"winner": "tie"})
    r2 = _parse_json(_safe_invoke(judge_llm, PAIRWISE_PROMPT.format(q=q, ctx=ctx, a=b, b=a)),
                     default={"winner": "tie"})
    r2_mapped = {"A": "B", "B": "A", "tie": "tie"}.get(r2.get("winner"), "tie")
    if r1["winner"] == r2_mapped:
        return {"winner": r1["winner"], "agreement": True,
                "reasons": [r1.get("reason"), r2.get("reason")]}
    return {"winner": "tie", "agreement": False,
            "reasons": [r1.get("reason"), r2.get("reason")]}

## 10. Pattern D — Reference-based judge
You have a curated gold answer; ask "semantically equivalent?"

In [ ]:
REFERENCE_PROMPT = """Compare the candidate ANSWER to the REFERENCE answer.
Are they semantically equivalent (same meaning, same key facts)?

Reply with ONLY a JSON object. "equivalent" must be a JSON boolean: true or false.
Example of the exact shape (do not copy the values):
{{"equivalent": true, "reason": "both answers convey the same steps"}}

Now compare:
QUESTION:  {q}
REFERENCE: {ref}
ANSWER:    {a}
"""

def judge_reference(q, ref, a):
    raw = _safe_invoke(judge_llm, REFERENCE_PROMPT.format(q=q, ref=ref, a=a))
    return _parse_json(raw, default={"equivalent": False, "reason": "parse error"})

## 11. Pattern E ★ — Reference-FREE faithfulness
**The killer pattern for production RAG.** No ground truth needed — only the retrieved context. Extracts atomic claims, verifies each against context.

In [ ]:
FAITHFULNESS_PROMPT = """You are checking whether an ANSWER is FAITHFUL to the CONTEXT.
An answer is faithful iff EVERY factual claim in it is supported by the CONTEXT.
Opinions, generic pleasantries, and restatements of the question are exempt.

Steps:
 1. Extract atomic factual claims from the ANSWER.
 2. For each claim, mark it supported or not supported by the CONTEXT.
 3. Reply with ONLY a JSON object. "supported" and "faithful" must be JSON booleans (true / false).

Example of the exact shape (do not copy the values):
{{"claims": [{{"claim": "the plan costs $10", "supported": true}}, {{"claim": "free trial available", "supported": false}}], "faithful": false, "reason": "one claim is not in the context"}}

Now evaluate:
CONTEXT: {ctx}
ANSWER:  {a}
"""

def judge_faithfulness(ctx, a):
    raw = _safe_invoke(judge_llm, FAITHFULNESS_PROMPT.format(ctx=ctx, a=a))
    return _parse_json(raw, default={"faithful": False, "claims": [], "reason": "parse error"})

## 12. Pipeline — generate v1 + v2, then judge with all patterns

In [ ]:
@dataclass
class Row:
    qid: str
    question: str
    answer_v1: str
    answer_v2: str
    pf_v1: str
    pf_v2: str
    rubric_v1: dict
    rubric_v2: dict
    pairwise: dict
    ref_eq_v1: bool
    ref_eq_v2: bool


def run_full_eval(golden):
    rows = []
    for ex in golden:
        print("\n" + "-"*60)
        print(f"QUESTION [{ex['id']}]: {ex['question']}")
        print("-"*60)
        a1 = gen_v1(ex["question"], ex["context"])
        a2 = gen_v2(ex["question"], ex["context"])
        print(f"   v1: {a1[:120]}")
        print(f"   v2: {a2[:120]}")

        pf1 = judge_passfail(ex["question"], ex["context"], a1)["verdict"]
        pf2 = judge_passfail(ex["question"], ex["context"], a2)["verdict"]
        rb1 = judge_rubric(ex["question"], ex["context"], a1)
        rb2 = judge_rubric(ex["question"], ex["context"], a2)
        pw  = judge_pairwise(ex["question"], ex["context"], a1, a2)
        rf1 = judge_reference(ex["question"], ex["ground_truth"], a1)["equivalent"]
        rf2 = judge_reference(ex["question"], ex["ground_truth"], a2)["equivalent"]

        rows.append(Row(ex["id"], ex["question"], a1, a2, pf1, pf2,
                        rb1, rb2, pw, rf1, rf2))
        print(f"   pass/fail   v1={pf1}  v2={pf2}")
        print(f"   pairwise    winner={pw['winner']}   agreement={pw['agreement']}")
        print(f"   ref-equiv   v1={rf1}  v2={rf2}")
    return rows

## 13. Reports — calibration, stability, regression, ship decision

In [ ]:
def calibration_report(rows, golden):
    print("\n" + "="*70 + "\n CALIBRATION  (judge vs human pass/fail on v2)\n" + "="*70)
    matches = sum(1 for r, ex in zip(rows, golden) if r.pf_v2 == ex["human_label_passfail"])
    n = len(rows)
    pct = 100 * matches / n
    print(f"   Simple agreement: {matches}/{n}  ({pct:.0f}%)")
    print(f"   Rule of thumb: < 80% -> re-tune rubric or use a stronger judge.")


def stability_check(rows, golden, n_runs=3):
    print("\n" + "="*70 + "\n JUDGE STABILITY  (same input x N runs)\n" + "="*70)
    sample = next(r for r in rows if r.qid == "q1")
    ctx = next(ex["context"] for ex in golden if ex["id"] == sample.qid)
    scores = []
    for i in range(n_runs):
        r = judge_rubric(sample.question, ctx, sample.answer_v2)
        f = r["scores"].get("faithfulness", 0)
        scores.append(f)
        print(f"   run {i+1}: faithfulness = {f}")
    sd = statistics.pstdev(scores)
    print(f"   Mean = {statistics.mean(scores):.2f}     StdDev = {sd:.2f}     (target: < 0.6)")
    print("   [warn] Judge UNSTABLE" if sd > 0.6 else "   [OK] Judge stable enough to trust")


def regression_report(rows):
    print("\n" + "="*70 + "\n REGRESSION REPORT  (v1 -> v2)\n" + "="*70)
    pf1 = sum(r.pf_v1 == "pass" for r in rows) / len(rows)
    pf2 = sum(r.pf_v2 == "pass" for r in rows) / len(rows)
    print(f"   Pass-rate              v1={pf1:.0%}   v2={pf2:.0%}   d={pf2-pf1:+.0%}")
    rubric_means = {}
    for dim in ["faithfulness", "completeness", "relevance", "clarity"]:
        m1 = statistics.mean(r.rubric_v1["scores"].get(dim, 0) for r in rows)
        m2 = statistics.mean(r.rubric_v2["scores"].get(dim, 0) for r in rows)
        rubric_means[dim] = (m1, m2)
        print(f"   Rubric . {dim:13s}  v1={m1:.2f}  v2={m2:.2f}   d={m2-m1:+.2f}")
    wins_v1 = sum(r.pairwise["winner"] == "A" for r in rows)
    wins_v2 = sum(r.pairwise["winner"] == "B" for r in rows)
    ties    = sum(r.pairwise["winner"] == "tie" for r in rows)
    print(f"   Pairwise               v1={wins_v1}  v2={wins_v2}  ties={ties}")
    ref1 = sum(r.ref_eq_v1 for r in rows) / len(rows)
    ref2 = sum(r.ref_eq_v2 for r in rows) / len(rows)
    print(f"   Ref-equivalence rate   v1={ref1:.0%}   v2={ref2:.0%}   d={ref2-ref1:+.0%}")
    return {"pass_rate": (pf1, pf2), "rubric": rubric_means,
            "pairwise": (wins_v1, wins_v2, ties), "ref_equiv": (ref1, ref2)}


def ship_decision(metrics, total):
    pf1, pf2 = metrics["pass_rate"]
    f1, f2   = metrics["rubric"]["faithfulness"]
    print("\n" + "="*50 + "\nDECISION  (rule: dpass >= 0 AND dfaithfulness >= 0)\n" + "="*50)
    d_pass, d_faith = pf2 - pf1, f2 - f1
    if d_pass >= 0 and d_faith >= 0:
        print("[SHIP] Ship v2")
    else:
        reasons = []
        if d_pass  < 0: reasons.append(f"pass-rate dropped ({d_pass:+.2f})")
        if d_faith < 0: reasons.append(f"faithfulness dropped ({d_faith:+.2f})")
        print(f"[HOLD] Do NOT ship v2 - {'; '.join(reasons)}")

## 14. Failure-mode live demo — verbosity + position bias
Crafted A vs B where B is long and fluffy but says the same thing.

In [ ]:
def failure_mode_demo():
    banner("7  FAILURE MODES OF LLM JUDGES  (live)")
    q   = "How do I reset my password?"
    ctx = ("To reset your password, go to Settings -> Security -> Reset password. "
           "A verification email will be sent.")
    short_correct = "Go to Settings -> Security -> Reset password."
    long_fluffy   = ("Certainly! I'd be delighted to help you on your password-reset "
                     "journey. As a thoughtful assistant, let me first acknowledge "
                     "that password resets can feel stressful. You may proceed to "
                     "Settings, then Security, and from there select 'Reset password' "
                     "to receive a verification email so you can regain access "
                     "promptly and securely.")
    print("\n   A = short & correct;  B = long & fluffy (same facts).")
    r = judge_pairwise(q, ctx, short_correct, long_fluffy)
    print(f"   Winner (after position swap): {r['winner']}   agreement={r['agreement']}")
    if r["winner"] == "B":
        print("   [warn] VERBOSITY BIAS - add 'prefer concise; do not reward length' to rubric.")
    elif not r["agreement"]:
        print("   [warn] POSITION BIAS - judge flipped on swap; require agreement (already done).")
    else:
        print("   [OK] Judge resisted verbosity bias.")

## 15. Run the full demo
This is the cell that produces the headline output. ~3-5 min on Groq free tier with `QUICK=True`.

In [ ]:
try:
    QUICK
except NameError:
    raise NameError(
        "QUICK / GOLDEN / judge_llm are not defined yet.\n"
        "You skipped the setup cells. In Colab: menu -> Runtime -> Run before, "
        "then re-run this cell."
    )

print("="*70 + "\n  LLM-as-a-Judge tutorial - RAG support bot eval\n" + "="*70)

# QUICK mode -> first 2 items + the q6 trap
golden = (GOLDEN[:2] + [g for g in GOLDEN if g["id"] == "q6"]) if QUICK else GOLDEN

banner("4  EVALUATION DESIGN PATTERNS - running A-E on the golden set")
rows = run_full_eval(golden)

  LLM-as-a-Judge tutorial - RAG support bot eval

  4  EVALUATION DESIGN PATTERNS - running A-E on the golden set

------------------------------------------------------------
QUESTION [q1]: How do I reset my password?
------------------------------------------------------------
   v1: Go to Settings -> Security -> Reset password.
   v2: Go to Settings -> Security -> Reset password. A verification email will be sent.
   pass/fail   v1=fail  v2=pass
   pairwise    winner=B   agreement=True
   ref-equiv   v1=False  v2=True

------------------------------------------------------------
QUESTION [q2]: What's the price of the Enterprise plan?
------------------------------------------------------------
   v1: The Enterprise plan starts at $499/month.
   v2: The Enterprise plan starts at $499/month.
   pass/fail   v1=pass  v2=pass
   pairwise    winner=tie   agreement=True
   ref-equiv   v1=False  v2=False

------------------------------------------------------------
QUESTION [q6]: What is yo

## 16. Pattern E spotlight — reference-FREE faithfulness on v2
Watch q6 fail (context can't support a refund-policy claim).

In [ ]:
banner("4-E  REFERENCE-FREE FAITHFULNESS  (no ground truth needed)")
for ex, r in zip(golden, rows):
    f = judge_faithfulness(ex["context"], r.answer_v2)
    flag = "[OK]" if f.get("faithful") else "[NO]"
    print(f"   {flag} [{ex['id']}] faithful={f.get('faithful')}  reason={f.get('reason','')[:80]}")


  4-E  REFERENCE-FREE FAITHFULNESS  (no ground truth needed)
   [OK] [q1] faithful=True  reason=all claims are in the context
   [OK] [q2] faithful=True  reason=all claims are supported by the context
   [OK] [q6] faithful=True  reason=the answer does not contain any factual claims


## 17. Calibration + stability

In [ ]:
calibration_report(rows, golden)
stability_check(rows, golden)


 CALIBRATION  (judge vs human pass/fail on v2)
   Simple agreement: 2/3  (67%)
   Rule of thumb: < 80% -> re-tune rubric or use a stronger judge.

 JUDGE STABILITY  (same input x N runs)
   run 1: faithfulness = 5
   run 2: faithfulness = 5
   run 3: faithfulness = 5
   Mean = 5.00     StdDev = 0.00     (target: < 0.6)
   [OK] Judge stable enough to trust


## 18. Regression report + ship/no-ship decision
This is the CI gate from slide 25.

In [ ]:
metrics = regression_report(rows)
ship_decision(metrics, total=len(rows))


 REGRESSION REPORT  (v1 -> v2)
   Pass-rate              v1=67%   v2=100%   d=+33%
   Rubric . faithfulness   v1=5.00  v2=5.00   d=+0.00
   Rubric . completeness   v1=3.33  v2=3.33   d=+0.00
   Rubric . relevance      v1=5.00  v2=5.00   d=+0.00
   Rubric . clarity        v1=4.67  v2=5.00   d=+0.33
   Pairwise               v1=1  v2=1  ties=1
   Ref-equivalence rate   v1=0%   v2=67%   d=+67%

DECISION  (rule: dpass >= 0 AND dfaithfulness >= 0)
[SHIP] Ship v2


## 19. Failure-mode demo

In [ ]:
failure_mode_demo()
print("\n[Done]  Next: wire this into CI - fail the build when dpass < 0 or dfaithfulness < 0.")


  7  FAILURE MODES OF LLM JUDGES  (live)

   A = short & correct;  B = long & fluffy (same facts).
   Winner (after position swap): B   agreement=True
   [warn] VERBOSITY BIAS - add 'prefer concise; do not reward length' to rubric.

[Done]  Next: wire this into CI - fail the build when dpass < 0 or dfaithfulness < 0.
